# Clean
fixing bad rows

In [1]:
#read the csv file and turn it into a ddataframe
import pandas as pd

df = pd.read_csv('../data/train-test.csv')

print("Shape", df.shape)
display(df.head()) #display the first 5 rows of the dataframe

Shape (48000, 14)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [2]:
df.isnull().sum() #check for missing values in the dataframe

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

In [3]:
'''
There are missing values in 'weight' and 'market_index' columns.
I need to fill in the missing values with a median value
'''

#finding the median value of 'market_index' column
df.groupby("date")["market_index"].agg(
    ["count", "min", "median", "max"]
).head(10)



,count,min,median,max
date,,,,
2025-01-01,144,0.89719,0.962590,1.01366
2025-01-02,166,0.92791,0.989030,1.05003
2025-01-03,156,0.85930,0.941040,1.02190
2025-01-04,154,0.79511,0.855715,0.93062
2025-01-05,152,0.73250,0.800030,0.87638
2025-01-06,167,0.74138,0.813120,0.87228
2025-01-07,160,0.79760,0.888890,0.95473
2025-01-08,188,0.92266,0.971425,1.03368
2025-01-09,146,0.94411,0.998300,1.05631


In [4]:
#Finding the min, max and median of 'weight'
df["weight"].describe()

count    47700.000000
mean     31028.844004
std       9391.440620
min     -47500.000000
25%      25800.000000
50%      31436.500000
75%      37018.000000
max      47500.000000
Name: weight, dtype: float64

In [5]:
'''
There are missing values in 'weight' and I need to treat it
as a missing value. Before doing that, I need to calculate the
invalid weights
'''

negative_weights = (df["weight"] < 0).sum()     #Check for negative values
zero_weights = (df["weight"] == 0).sum()        #Check for zero values
missing_weights = df["weight"].isnull().sum()   #Check for missing values

print("Negative weights:", negative_weights)
print("Zero weights:", zero_weights)
print("Missing weights:", missing_weights)

Negative weights: 292
Zero weights: 0
Missing weights: 300


In [6]:
'''
If there are incorrect values with the 'weight' and 'market_index'
columns I am going to assume that there are incorrect values in
other columns, I need to check
'''

print("Invalid distances:", (df["distance"] <= 0).sum())
print("Invalid posted rates:", (df["posted_rate"] <= 0).sum())

display(df[["distance", "posted_rate"]].describe())


Invalid distances: 0
Invalid posted rates: 0


,distance,posted_rate
count,48000.000000,48000.000000
mean,1135.856654,2373.980682
std,728.564416,1486.493245
min,70.000000,57.220000
25%,550.400000,1251.555000
50%,953.300000,2030.760000
75%,1645.525000,3330.750000
max,3439.800000,25533.000000


In [7]:
'''
I want to check Rate Per Mile (Rate Per Mile = posted_rate / distance)
to see if there are any outliers in the data
'''


# Calculate the freight rate per mile for inspection.
df["rate_per_mile"] = df["posted_rate"] / df["distance"]

display(df["rate_per_mile"].describe())

display(df.nlargest(10, "rate_per_mile")[
    ["load_id", "distance", "equipment", "posted_rate", "rate_per_mile"]
])


count    48000.000000
mean         2.215347
std          0.583887
min          0.333709
25%          1.986974
50%          2.145348
75%          2.342696
max         14.125294
Name: rate_per_mile, dtype: float64

,load_id,distance,equipment,posted_rate,rate_per_mile
22254,TR-022255,314.7,Reefer,4445.23,14.125294
36609,TR-036610,98.4,Reefer,1386.42,14.089634
870,TR-000871,305.6,Reefer,4069.24,13.315576
17070,TR-017071,201.5,Flatbed,2609.34,12.949578
27969,TR-027970,589.9,Reefer,7597.52,12.879335
23369,TR-023370,412.4,Flatbed,5308.96,12.873327
28012,TR-028013,604.3,Reefer,7773.37,12.863429
45483,TR-045484,153.8,Dry Van,1971.21,12.816710
1415,TR-001416,198.2,Reefer,2509.96,12.663774
21968,TR-021969,254.5,Dry Van,3220.66,12.654853


In [8]:
'''
There are some unusual values here, I am unsure
if they are genuine expensive shipments due to 
the route or cargo type, or if they are incorrect values.
I need to figure out if these are isolated cases
or not
'''

# Count loads above different rate-per-mile thresholds.
print("Above $3/mile:", (df["rate_per_mile"] > 3).sum())
print("Above $5/mile:", (df["rate_per_mile"] > 5).sum())
print("Above $10/mile:", (df["rate_per_mile"] > 10).sum())

# Count unusually low rates.
print("Below $1/mile:", (df["rate_per_mile"] < 1).sum())


Above $3/mile: 772
Above $5/mile: 317
Above $10/mile: 49
Below $1/mile: 329


In [9]:
'''
I want to check for duplicate load_id
and duplicate rows
'''

print("Duplicate load IDs:", df["load_id"].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())


Duplicate load IDs: 0
Duplicate rows: 0


In [10]:
'''
Double checking the dates before I move on
'''


# Convert dates into datetime format
dates = pd.to_datetime(df["date"], errors="coerce")

# Check for dates Python couldn't understand
print("Invalid dates:", dates.isna().sum())

# Check the date range
print("Earliest date:", dates.min())
print("Latest date:", dates.max())

# Check for dates outside the training period
outside_range = (
    (dates < "2025-01-01") |
    (dates > "2025-10-31")
)

print("Dates outside expected range:", outside_range.sum())


Invalid dates: 0
Earliest date: 2025-01-01 00:00:00
Latest date: 2025-10-31 00:00:00
Dates outside expected range: 0


## Data cleaning summary

Most of the data looks good, but a few issues need fixing before modelling. I have enough information to move forward.

| Issue | Rows | Action |
|---|---|---|
| Missing weights | 300 | Fill using the training median |
| Negative weights | 292 | Treat as missing, then impute |
| Missing market index | 374 | Fill using the same-day median |
| Missing distances or posted rates | 0 | No action required |
| Unusual rates per mile | Not yet formally classified | investigate if model validation reveals problems |

# Cleaning
This section is for testing the functions inside cleaning.py

In [11]:
import sys

# Add the src directory to the Python path so I can import my cleaning functions
sys.path.append("../src")

from cleaning import clean_weight, clean_market_index

In [12]:
'''
I'm using the full dataset to check if the functions
work correctly. I will use the training dataset to train the model
'''

# Calculate the median using only valid weights
median_weight = df.loc[df["weight"] > 0, "weight"].median()

# Calculate the market index median for each day
daily_medians = df.groupby("date")["market_index"].median()

# Calculate the overall market index median as a fallback
overall_median = df["market_index"].median()

print("Median weight:", median_weight)
print("Overall market index median:", overall_median)


Median weight: 31493.5
Overall market index median: 1.0558


In [13]:
# Apply the cleaning functions
cleaned_df = clean_weight(df, median_weight)
cleaned_df = clean_market_index(cleaned_df, daily_medians, overall_median)

# Verify the results
print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))
print("Invalid weights:", (cleaned_df["weight"] <= 0).sum())
print("Missing weights:", cleaned_df["weight"].isna().sum())
print("Missing market indices:", cleaned_df["market_index"].isna().sum())


Original rows: 48000
Cleaned rows: 48000
Invalid weights: 0
Missing weights: 0
Missing market indices: 0


In [14]:
'''
Checking to see if the create_features
function works correctly'''

from features import create_features

X = create_features(cleaned_df)

print(X.shape)
display(X.head())

(48000, 14)


,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,market_index,quote_signal,month,day_of_week,days_since_start
0,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,0.95684,2.39595,1,2,0
1,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,0.97623,2.43355,1,2,0
2,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,1.00971,1.84491,1,2,0
3,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,0.94518,1.87712,1,2,0
4,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,0.98480,2.56300,1,2,0


# Train Test Split

In [15]:
'''
Setting the train test split to 80/20
training on Jan-Aug and testing on 
Sep-Oct
'''

# Convert dates to datetime for comparison
df["date"] = pd.to_datetime(df["date"])

# Split by date
train_df = df[df["date"] < "2025-09-01"].copy()
test_df = df[df["date"] >= "2025-09-01"].copy()

# Check the split
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("Training dates:", train_df["date"].min(), "to", train_df["date"].max())
print("Test dates:", test_df["date"].min(), "to", test_df["date"].max())


Training rows: 38477
Test rows: 9523
Training dates: 2025-01-01 00:00:00 to 2025-08-31 00:00:00
Test dates: 2025-09-01 00:00:00 to 2025-10-31 00:00:00


In [16]:
# Calculate cleaning values from training data ONLY
# except for the daily medians
median_weight = train_df.loc[
    train_df["weight"] > 0, "weight"
].median()

daily_medians = df.groupby("date")["market_index"].median()

overall_median = train_df["market_index"].median()

# Clean the training data
train_clean = clean_weight(train_df, median_weight)
train_clean = clean_market_index(train_clean, daily_medians, overall_median)

# Clean the holdout data using the SAME training medians
test_clean = clean_weight(test_df, median_weight)
test_clean = clean_market_index(test_clean, daily_medians, overall_median)

print("Training rows:", len(train_clean))
print("Test rows:", len(test_clean))

print("Training missing values:")
print(train_clean[["weight", "market_index"]].isna().sum())

print("Test missing values:")
print(test_clean[["weight", "market_index"]].isna().sum())

Training rows: 38477
Test rows: 9523
Training missing values:
weight          0
market_index    0
dtype: int64
Test missing values:
weight          0
market_index    0
dtype: int64


# Compare Models
note to self: X = the info the model uses to predict freight rates
Y = the actual freigh rate prices which the model learns to predict

In [17]:
# Prepare model inputs
X_train = create_features(train_clean)
X_test = create_features(test_clean)

# Prepare the target (actual freight rates)
y_train = train_clean["posted_rate"]
y_test = test_clean["posted_rate"]

# Check the shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (38477, 14)
X_test: (9523, 14)
y_train: (38477,)
y_test: (9523,)


In [18]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

'''
Experiment 1
this has 14 features
'''
# Tell CatBoost which columns contain categories
categorical_features = ["pickup", "delivery", "equipment"]

# Create our model
model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False,
    random_seed=42
)

# Train using January–August data
model.fit(
    X_train,
    y_train,
    cat_features=categorical_features
)

# Predict September–October freight rates
predictions = model.predict(X_test)

# Calculate prediction errors
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"MAE: ${mae:.2f}")
print(f"RMSE: ${rmse:.2f}")

MAE: $122.16
RMSE: $636.34


In [ ]:
'''
Remove two date features for Experiment 2 
this as 12 features, I dropped months and days since start
'''

X_train_2 = X_train.drop(columns=["month", "days_since_start"])
X_test_2 = X_test.drop(columns=["month", "days_since_start"])

# Train a new CatBoost model with the same settings
model_2 = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False,
    random_seed=42
)

model_2.fit(
    X_train_2,
    y_train,
    cat_features=categorical_features
)

# Evaluate Experiment 2
predictions_2 = model_2.predict(X_test_2)

mae_2 = mean_absolute_error(y_test, predictions_2)
rmse_2 = np.sqrt(mean_squared_error(y_test, predictions_2))

print(f"Experiment 2 MAE: ${mae_2:.2f}")
print(f"Experiment 2 RMSE: ${rmse_2:.2f}")

Experiment 2 MAE: $135.06
Experiment 2 RMSE: $640.76


In [20]:
'''
Experiment 3
Linear Regression (distance only)
'''
from sklearn.linear_model import LinearRegression

# Use distance as our only feature
X_train_linear = X_train[["distance"]]
X_test_linear = X_test[["distance"]]

# Create and train the baseline
linear_model = LinearRegression()
linear_model.fit(X_train_linear, y_train)

# Predict September–October freight rates
linear_predictions = linear_model.predict(X_test_linear)

# Calculate baseline prediction errors
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))

print(f"Linear Regression MAE: ${linear_mae:.2f}")
print(f"Linear Regression RMSE: ${linear_rmse:.2f}")

Linear Regression MAE: $196.95
Linear Regression RMSE: $654.42


In [21]:
'''
I want to seperate the test loads and typical priced loads to see how the models perform on the typical-priced loads
'''
# Calculate actual rate per mile for the holdout loads
test_rate_per_mile = y_test / X_test["distance"]

# Identify loads priced between $1 and $5 per mile
normal_mask = test_rate_per_mile.between(1, 5)

print("Total test loads:", len(y_test))
print("Typical-priced loads:", normal_mask.sum())
print("Unusual-priced loads:", (~normal_mask).sum())

# Evaluate each model on typical-priced test loads
for name, predictions_to_check in [
    ("CatBoost - All Features", predictions),
    ("CatBoost - Day of Week Only", predictions_2),
    ("Linear Regression", linear_predictions)
]:
    typical_mae = mean_absolute_error(
        y_test[normal_mask],
        predictions_to_check[normal_mask]
    )

    typical_rmse = np.sqrt(mean_squared_error(
        y_test[normal_mask],
        predictions_to_check[normal_mask]
    ))

    print(f"{name}")
    print(f"  Typical MAE: ${typical_mae:.2f}")
    print(f"  Typical RMSE: ${typical_rmse:.2f}")



Total test loads: 9523
Typical-priced loads: 9386
Unusual-priced loads: 137
CatBoost - All Features
  Typical MAE: $69.86
  Typical RMSE: $171.93
CatBoost - Day of Week Only
  Typical MAE: $82.98
  Typical RMSE: $180.60
Linear Regression
  Typical MAE: $145.61
  Typical RMSE: $231.54


In [22]:
'''
Experiment 4
CatBoost without unusual training prices
'''

# Calculate the actual rate per mile in training
train_rate_per_mile = y_train / X_train["distance"]

# Select training loads between $1 and $5 per mile
train_mask = train_rate_per_mile.between(1, 5)

# Create filtered training data
X_train_filtered = X_train.loc[train_mask]
y_train_filtered = y_train.loc[train_mask]

print("Original training rows:", len(X_train))
print("Filtered training rows:", len(X_train_filtered))
print("Excluded training rows:", (~train_mask).sum())

# Same model settings as Experiment 1
model_filtered = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False,
    random_seed=42
)

# Train using only typical-priced training loads
model_filtered.fit(
    X_train_filtered,
    y_train_filtered,
    cat_features=categorical_features
)

# Predict ALL September–October loads
predictions_filtered = model_filtered.predict(X_test)

# Evaluate using all holdout loads
filtered_mae = mean_absolute_error(y_test, predictions_filtered)
filtered_rmse = np.sqrt(mean_squared_error(y_test, predictions_filtered))

print("Original CatBoost:")
print(f"  MAE: ${mae:.2f}")
print(f"  RMSE: ${rmse:.2f}")

print("\nFiltered CatBoost:")
print(f"  MAE: ${filtered_mae:.2f}")
print(f"  RMSE: ${filtered_rmse:.2f}")


Original training rows: 38477
Filtered training rows: 37968
Excluded training rows: 509
Original CatBoost:
  MAE: $122.16
  RMSE: $636.34

Filtered CatBoost:
  MAE: $118.01
  RMSE: $636.95


In [23]:

# Compare all four experiments
results = pd.DataFrame({
    "Model": [
        "CatBoost - All Features",
        "CatBoost - Day of Week Only (Date Features)",
        "Linear Regression - Distance Only",
        "CatBoost - Filtered Training Data"
    ],
    "MAE": [mae, mae_2, linear_mae, filtered_mae],
    "RMSE": [rmse, rmse_2, linear_rmse, filtered_rmse]
})

display(results)

# Save the comparison
results.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)


,Model,MAE,RMSE
0,CatBoost - All Features,122.159796,636.344330
1,CatBoost - Day of Week Only (Date Features),135.058299,640.755115
2,Linear Regression - Distance Only,196.950147,654.415640
3,CatBoost - Filtered Training Data,118.010750,636.945932


# Refit

In [ ]:
'''
Prepare the full development dataset
'''

full_df = df.copy()

# Calculate cleaning statistics using all labeled data
median_weight = full_df.loc[
    full_df["weight"] > 0, "weight"
].median()

daily_medians = full_df.groupby("date")["market_index"].median()
overall_median = full_df["market_index"].median()

# Apply our existing cleaning functions
full_clean = clean_weight(full_df, median_weight)
full_clean = clean_market_index(
    full_clean, daily_medians, overall_median
)

print("Original rows:", len(full_df))
print("Cleaned rows:", len(full_clean))
print("Missing weights:", full_clean["weight"].isna().sum())
print("Missing market indices:", full_clean["market_index"].isna().sum())

# Create features and target
X_full = create_features(full_clean)
y_full = full_clean["posted_rate"]

# Calculate rate per mile for filtering training rows
full_rate_per_mile = y_full / X_full["distance"]

# Keep training loads between $1 and $5 per mile
full_mask = full_rate_per_mile.between(1, 5)

X_final = X_full.loc[full_mask]
y_final = y_full.loc[full_mask]

print("Original training rows:", len(X_full))
print("Final training rows:", len(X_final))
print("Excluded rows:", (~full_mask).sum())
print("Features:", X_final.shape[1])

# Create the final model using our chosen settings
final_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False,
    random_seed=42
)

# Train using all eligible January–October loads
final_model.fit(
    X_final,
    y_final,
    cat_features=categorical_features
)

print("Final CatBoost model trained successfully!")

Original rows: 48000
Cleaned rows: 48000
Missing weights: 0
Missing market indices: 0
Original training rows: 48000
Final training rows: 47354
Excluded rows: 646
Features: 14
Final CatBoost model trained successfully!


In [27]:
'''
Save the model
'''

final_model.save_model("../models/freight_catboost.cbm")

print("Model saved successfully!")


Model saved successfully!


# Load and inspect prediction data

In [28]:
# Load the 12,000 loads requiring predictions
validation_df = pd.read_csv("../data/validation.csv")

# Check the dataset
print("Validation shape:", validation_df.shape)
print("Unique load IDs:", validation_df["load_id"].nunique())

# Check missing values
print("\nMissing values:")
print(validation_df.isna().sum())

# Preview the data
display(validation_df.head())

Validation shape: (12000, 13)
Unique load IDs: 12000

Missing values:
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [29]:
# Convert dates to the same format used during training
validation_df["date"] = pd.to_datetime(validation_df["date"])

# Calculate daily medians from available validation market values
validation_daily_medians = validation_df.groupby("date")["market_index"].median()

# Apply our existing cleaning functions
validation_clean = clean_weight(validation_df, median_weight)

validation_clean = clean_market_index(
    validation_clean,
    validation_daily_medians,
    overall_median
)

# Verify cleaning
print("Original rows:", len(validation_df))
print("Cleaned rows:", len(validation_clean))
print("Missing weights:", validation_clean["weight"].isna().sum())
print("Missing market indices:", validation_clean["market_index"].isna().sum())


Original rows: 12000
Cleaned rows: 12000
Missing weights: 0
Missing market indices: 0


In [31]:
# Create features for the 12,000 validation loads
X_validation = create_features(validation_clean)

# Match the feature order used during training
X_validation = X_validation[X_final.columns]

# Check the features
print("Validation shape:", X_validation.shape)
print("Features match:", X_validation.columns.equals(X_final.columns))

# Predict freight rates for all 12,000 loads
validation_predictions = final_model.predict(X_validation)

# Check the results
print("Number of predictions:", len(validation_predictions))
print("First five predictions:", validation_predictions[:5])


Validation shape: (12000, 14)
Features match: True
Number of predictions: 12000
First five predictions: [ 863.20903604 5073.30782721 5280.34594452 4145.64482243 1861.24977307]


In [33]:
'''
Associate predicted price with its
corrisponding load_id
'''

# Load the provided submission template
submission = pd.read_csv("../data/validation-predictions-template.csv")

# Match predictions to their load IDs
prediction_lookup = pd.Series(
    validation_predictions,
    index=validation_df["load_id"]
)

submission["predicted_rate"] = submission["load_id"].map(prediction_lookup)

# Save the completed CSV
submission.to_csv(
    "../outputs/validation_predictions.csv",
    index=False
)

# Preview the submission
print("Submission shape:", submission.shape)
display(submission.head())

Submission shape: (12000, 2)


,load_id,predicted_rate
0,TE-000001,863.209036
1,TE-000002,5073.307827
2,TE-000003,5280.345945
3,TE-000004,4145.644822
4,TE-000005,1861.249773


# Validate the submission

In [34]:
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Duplicate IDs:", submission["load_id"].duplicated().sum())
print("Missing predictions:", submission["predicted_rate"].isna().sum())
print("Non-positive predictions:", (submission["predicted_rate"] <= 0).sum())
print("Non-finite predictions:", (~np.isfinite(submission["predicted_rate"])).sum())

Rows: 12000
Columns: ['load_id', 'predicted_rate']
Duplicate IDs: 0
Missing predictions: 0
Non-positive predictions: 0
Non-finite predictions: 0


In [36]:
'''
Load dataset paths for december
'''

# Load the December chart inputs
december_df = pd.read_csv("../data/december-chart-inputs.csv")

print("Shape:", december_df.shape)
print("Columns:", december_df.columns.tolist())

print("\nMissing values:")
print(december_df.isna().sum())

display(december_df.head())

Shape: (31, 7)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

Missing values:
pickup             0
delivery           0
distance           0
equipment          0
weight             0
date               0
predicted_rate    31
dtype: int64


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


In [37]:
'''
My CatBoost model uses latitude and longitude as features, 
but december_chart_inputs.csv doesn't contain them
'''

print("Lexington pickup loads:",
      (full_df["pickup"] == "Lexington").sum())

print("Fort Wayne delivery loads:",
      (full_df["delivery"] == "Fort Wayne").sum())


Lexington pickup loads: 1209
Fort Wayne delivery loads: 1176


In [38]:
'''
Extract the co-ords
'''

# Get Lexington's pickup coordinates
lexington_coords = full_df.loc[
    full_df["pickup"] == "Lexington",
    ["pickup_lat", "pickup_lon"]
].median()

# Get Fort Wayne's delivery coordinates
fort_wayne_coords = full_df.loc[
    full_df["delivery"] == "Fort Wayne",
    ["delivery_lat", "delivery_lon"]
].median()

print("Lexington coordinates:")
print(lexington_coords)

print("\nFort Wayne coordinates:")
print(fort_wayne_coords)


Lexington coordinates:
pickup_lat    36.99152
pickup_lon   -84.99876
dtype: float64

Fort Wayne coordinates:
delivery_lat    41.31561
delivery_lon   -85.36206
dtype: float64


In [42]:
'''
Add the missing features
'''

# Make a copy of the December inputs
december_ready = december_df.copy()

# Add coordinates from our training data
december_ready["pickup_lat"] = lexington_coords["pickup_lat"]
december_ready["pickup_lon"] = lexington_coords["pickup_lon"]

december_ready["delivery_lat"] = fort_wayne_coords["delivery_lat"]
december_ready["delivery_lon"] = fort_wayne_coords["delivery_lon"]

# Calculate daily market medians from December validation data
validation_df["date"] = pd.to_datetime(validation_df["date"])

daily_market = validation_df[
    validation_df["date"].dt.month == 12
].groupby("date")[["market_index", "quote_signal"]].median()

# Match December dates to their market values
december_ready["date"] = pd.to_datetime(december_ready["date"])

december_ready["market_index"] = (
    december_ready["date"].map(daily_market["market_index"])
)

december_ready["quote_signal"] = (
    december_ready["date"].map(daily_market["quote_signal"])
)

# Create the same 14 features used by CatBoost
X_december = create_features(december_ready)
X_december = X_december[X_final.columns]

# Verify
print("December shape:", X_december.shape)
print("Features match:", X_december.columns.equals(X_final.columns))
print("Missing values:", X_december.isna().sum().sum())

December shape: (31, 14)
Features match: True
Missing values: 0


In [43]:
'''
Generate and save december predictions
'''
# Predict all 31 December freight rates
december_predictions = final_model.predict(X_december)

# Fill the original December template
december_output = december_df.copy()
december_output["predicted_rate"] = december_predictions

# Save the completed file
december_output.to_csv(
    "../outputs/december_predictions.csv",
    index=False
)

# Check the results
print("December rows:", len(december_output))
print("Missing predictions:", december_output["predicted_rate"].isna().sum())

display(december_output.head())


December rows: 31
Missing predictions: 0


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,816.496821
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,819.602966
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,829.099562
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,831.561489
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,827.027500


# Validating further
I found an error, before moving back and chaning things I want to test

In [41]:
# Examine market conditions for December dates
validation_df["date"] = pd.to_datetime(validation_df["date"])

december_market = validation_df[
    validation_df["date"].dt.month == 12
]

print("December validation loads:", len(december_market))
print("Unique December dates:", december_market["date"].nunique())

display(
    december_market.groupby("date")[
        ["market_index", "quote_signal"]
    ].median().head(10)
)

December validation loads: 6164
Unique December dates: 31


,market_index,quote_signal
date,,
2025-12-01,0.833480,2.058610
2025-12-02,0.914660,2.040490
2025-12-03,0.994420,2.036940
2025-12-04,1.025015,2.066700
2025-12-05,0.974900,2.067520
2025-12-06,0.889795,2.044125
2025-12-07,0.829470,2.039770
2025-12-08,0.840010,2.020340
2025-12-09,0.915950,2.060910
